# Analisi delle dipendenze API YAML

**Author:** *Francesco Pezzuto*

## Glossario

- **Overview**: breve descrizione del contenuto del notebook
- **DataFrame**: struttura tabellare usata per analizzare i dati
- **Grafo**: insieme di nodi e archi che rappresentano relazioni tra componenti
- **Dipendenza**: relazione tra due elementi del sistema, ad esempio tra file o tra schemi
- **Peso della dipendenza**: numero di occorrenze con cui una dipendenza compare

# Overview

Il notebook analizza le dipendenze estratte da un insieme di specifiche OpenAPI in formato YAML.
L'obiettivo principale è quello di analizzare le dipendenze tra file, schemi, identificare componenti centrali per preparare i dati per la successiva fase di clustering e modularizzazione

## Tecnologie utilizzate

Nel presente lavoro lo sviluppo della pipeline e' stato effettuato in Kotlin, mantenendo coerenza tra fase implementativa e fase di analisi.

Per la parte di parsing e' stata scelta la libreria Jackson. Prima di convergere su questa soluzione sono state considerate anche alternative come SnakeYAML o parser YAML piu' leggeri, ma Jackson e' risultato piu' adatto al caso di studio. Il motivo principale e' che il progetto richiede una lettura robusta di file OpenAPI complessi e non sempre uniformi, con necessita' di attraversare strutture annidate e individuare riferimenti `$ref` in modo ricorsivo. In questo contesto, la rappresentazione ad albero tramite `JsonNode` consente una navigazione flessibile senza imporre un modello rigido a priori.

In particolare sono stati utilizzati `jackson-dataformat-yaml` per la lettura dei file YAML e `jackson-module-kotlin` per migliorare l'integrazione con Kotlin. Questa combinazione ha reso il parser piu' stabile, piu' semplice da estendere e piu' adatto a gestire casi reali con varianti strutturali.

La fase di analisi viene condotta in notebook mediante Kotlin DataFrame. Questa scelta e' coerente con lo stack adottato e permette di mantenere un unico linguaggio lungo tutto il flusso, dalla generazione dei dati alla loro esplorazione. Attraverso DataFrame e' possibile leggere i CSV prodotti dalla pipeline, effettuare operazioni di filtraggio e aggregazione e calcolare metriche descrittive utili all'interpretazione del grafo delle dipendenze.

### Origine dei dati

I dati del seguente file sono stati generati da una sequenza pipeline sviluppata in Kotlin che parsa i file YAML, estrae le ref, costruisce le dipendenze e calcola i pesi (numero di occorenze) di ogni dipendenza. Esporta i risultati in formato CSV

In [11]:
%useLatestDescriptors
%use dataframe(1.0.0-Beta4n), kandy(0.8.3, 0.8.3)

In [12]:
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.io.*

val dependencies = DataFrame.readCSV(
    "C:\\Users\\Francesco.Pezzuto\\Desktop\\project_yaml\\src\\main\\kotlin\\com\\example\\demo\\output\\dependencies.csv"
)

val nodeStats = DataFrame.readCSV(
    "C:\\Users\\Francesco.Pezzuto\\Desktop\\project_yaml\\src\\main\\kotlin\\com\\example\\demo\\output\\nodeStats.csv"
)

println(dependencies.columnNames())
dependencies.head()

[from, to, type, count]


from,to,type,count
core.yaml,jsonapi.yaml,file_ref,387
items.yaml,jsonapi.yaml,file_ref,325
salesDoc.yaml,jsonapi.yaml,file_ref,194
surveyDoc.yaml,jsonapi.yaml,file_ref,126
purchaseDoc.yaml,jsonapi.yaml,file_ref,117


## Dataset: dependencies

Il dataset `dependencies` contiene le dipendenze estratte dalla pipeline.

Le colonne principali sono:
- `from`: nodo sorgente
- `to`: nodo destinazione
- `type`: tipo di dipendenza
- `count`: peso della dipendenza

Di seguito una breve illustrazione


In [13]:
dependencies

from,to,type,count
core.yaml,jsonapi.yaml,file_ref,387
items.yaml,jsonapi.yaml,file_ref,325
salesDoc.yaml,jsonapi.yaml,file_ref,194
surveyDoc.yaml,jsonapi.yaml,file_ref,126
purchaseDoc.yaml,jsonapi.yaml,file_ref,117
hierarchies.yaml,jsonapi.yaml,file_ref,107
customers.yaml,jsonapi.yaml,file_ref,102
tasks.yaml,jsonapi.yaml,file_ref,101
accessControl.yaml,jsonapi.yaml,file_ref,100
clusters.yaml,jsonapi.yaml,file_ref,95


## Dataset: nodeStats

Il dataset `nodeStats` contiene statistiche aggregate sui nodi del grafo delle dipendenze.

In particolare permette di osservare:
- grado in uscita
- grado in entrata
- centralità di base dei nodi

Di seguito una breve illustrazione

In [14]:
nodeStats

node,outDegree,inDegree,total
core.yaml,438,619,1057
jsonapi.yaml,2,3005,3007
items.yaml,426,63,489
salesDoc.yaml,279,33,312
surveyDoc.yaml,153,25,178
purchaseDoc.yaml,197,28,225
hierarchies.yaml,133,19,152
customers.yaml,172,28,200
tasks.yaml,116,22,138
accessControl.yaml,116,22,138


## Prime dipendenze più rilevanti

Di seguito vengono mostrate le dipendenze con peso maggiore.
Queste relazioni sono particolarmente interessanti perché indicano legami forti tra i componenti.

In [10]:
%use dataframe
import org.jetbrains.kotlinx.dataframe.api.*
import org.jetbrains.kotlinx.dataframe.io.*
import java.nio.file.Paths

val outputDir = Paths.get("../../output")
println(outputDir.toAbsolutePath())

val dependenciesPath = outputDir.resolve("dependencies.csv").toString()
val nodeStatsPath = outputDir.resolve("nodeStats.csv").toString()

val dependencies = DataFrame.readCSV(dependenciesPath)
val nodeStats = DataFrame.readCSV(nodeStatsPath)

println("dependencies path: $dependenciesPath")
println("nodeStats path: $nodeStatsPath")

C:\Users\michele.mauro\work\lab\demo\src\main\..\..\output
dependencies path: ..\..\output\dependencies.csv
nodeStats path: ..\..\output\nodeStats.csv


## Dataset: dependencies

Il dataset `dependencies` contiene le dipendenze estratte dalla pipeline.

Le colonne principali sono:
- `from`: nodo sorgente
- `to`: nodo destinazione
- `type`: tipo di dipendenza
- `count`: peso della dipendenza

Di seguito una breve illustrazione

In [11]:
dependencies

from,to,type,count
core.yaml,jsonapi.yaml,file_ref,387
items.yaml,jsonapi.yaml,file_ref,325
salesDoc.yaml,jsonapi.yaml,file_ref,194
surveyDoc.yaml,jsonapi.yaml,file_ref,126
purchaseDoc.yaml,jsonapi.yaml,file_ref,117
hierarchies.yaml,jsonapi.yaml,file_ref,107
customers.yaml,jsonapi.yaml,file_ref,102
tasks.yaml,jsonapi.yaml,file_ref,101
accessControl.yaml,jsonapi.yaml,file_ref,100
clusters.yaml,jsonapi.yaml,file_ref,95


## Distribuzione delle dipendenze per tipo

Questa sezione mostra quanti archi appartengono a ciascuna categoria di dipendenza.

In [2]:
nodeStats

org.jetbrains.kotlinx.jupyter.exceptions.ReplCompilerException: at Cell In[2], line 1, column 1: Unresolved reference: nodeStats

## Nodi più centrali

Questa tabella evidenzia i nodi con maggiore centralità, utile per identificare componenti condivisi o fortemente accoppiati.

In [17]:
nodeStats.sortByDesc("total").head(10)

node,outDegree,inDegree,total
jsonapi.yaml,2,3005,3007
core.yaml,438,619,1057
jsonapi.yaml::failure,4,631,635
openapi.yaml,576,0,576
items.yaml,426,63,489
salesDoc.yaml,279,33,312
documents.yaml,132,160,292
purchaseDoc.yaml,197,28,225
jsonapi.yaml::sort,0,201,201
customers.yaml,172,28,200


## Nodi InDegree più rilevanti

In [18]:
nodeStats.sortByDesc("inDegree").head(10)

node,outDegree,inDegree,total
jsonapi.yaml,2,3005,3007
jsonapi.yaml::failure,4,631,635
core.yaml,438,619,1057
jsonapi.yaml::sort,0,201,201
jsonapi.yaml::include,0,198,198
jsonapi.yaml::pageSize,0,186,186
jsonapi.yaml::pageNumber,0,186,186
jsonapi.yaml::acceptLanguage,0,186,186
dataTypes.yaml,0,163,163
documents.yaml,132,160,292


Si noti che in questo caso le chiamate interne più richieste sono fatte dal file jsonapi.yaml, insieme ai failure creati dal file stesso. Ne consegue una possibile dipendenza interna importante.

## Nodi OutDegree più rilevanti

In [19]:
nodeStats.sortByDesc("outDegree").head(10)

node,outDegree,inDegree,total
openapi.yaml,576,0,576
core.yaml,438,619,1057
items.yaml,426,63,489
salesDoc.yaml,279,33,312
purchaseDoc.yaml,197,28,225
customers.yaml,172,28,200
surveyDoc.yaml,153,25,178
warehouseDoc.yaml,151,21,172
hierarchies.yaml,133,19,152
documents.yaml,132,160,292


Invece per quanto riguarda l'analisi delle chiamate esterne effettuate alle API YAML, si noti che la più referenziata è il file openapi.


# Grafico per controllare le dipendenze